In [1]:
import sys

model_path = '/Users/rdobrien/skysurf/Zodiacal_Light_Model_Python'
if model_path not in sys.path:
    sys.path.append(model_path)

In [2]:
# # make_baseline.py -- run with the CURRENT get_zmod.py, before updating it
# import numpy as np
# from zodisurf.get_zmod import get_zmod

# N = 6
# DAY = np.linspace(100.0, 300.0, N)
# LON = np.linspace(20.0, 340.0, N)
# LAT = np.linspace(-60.0, 60.0, N)

# out = {}
# for lam in [1.25, 2.2, 3.5, 12.0, 25.0]:          # DIRBE bands
#     out[f'kelsall_{lam}'] = get_zmod(lam, 'kelsall', DAY, LON, LAT)
# for lam in [1.0, 1.25, 2.0, 3.0]:
#     out[f'skysurf_{lam}'] = get_zmod(lam, 'skysurf', DAY, LON, LAT)
# np.savez('zmod_baseline.npz', **out)
# print("saved", list(out))

In [3]:
"""
Tests for get_zmod, including phase_type='fitting'.
Run with:  pytest -v test_get_zmod_fitting.py
"""
import os
import numpy as np
import pytest

import zodisurf.get_zmod as gz
from zodisurf.get_zmod import get_zmod, _apply_fitting, _broadcast_fitting
from zodisurf.skysurf_params import (get_albedo, get_hong_params, get_mult,
                                     get_emiss, put_zpar)
from zodisurf.solar_sp import solar_sp

# Geometry must match make_baseline.py
N = 6
DAY = np.linspace(100.0, 300.0, N)
LON = np.linspace(20.0, 340.0, N)
LAT = np.linspace(-60.0, 60.0, N)

LAM = 1.25
HONG = np.array([0.7, -0.2, 0.1, 0.6, 0.3, 0.1])
SIRR = float(np.ravel(solar_sp(np.array([LAM])))[0])   # test input only

ALB_SLOTS = [33, 59, 85, 111, 137, 170]
EM_SMOOTH, EM_BANDS, EM_RING = [37], [63, 89, 115, 141], [174]
FIT_NAMES = ['fitting_hong', 'fitting_albedo', 'fitting_em_smooth',
             'fitting_em_band', 'fitting_em_ring']


def fit_kw(**over):
    kw = dict(solar_irr=SIRR, fitting_hong=HONG, fitting_albedo=0.25,
              fitting_em_smooth=0.9, fitting_em_band=0.9, fitting_em_ring=0.9)
    kw.update(over)
    return kw


def zf(lam=LAM, **over):
    return get_zmod(lam, 'fitting', DAY, LON, LAT, **fit_kw(**over))


# ---------------------------------------------------------------- 1. errors
@pytest.mark.parametrize('name', FIT_NAMES + ['solar_irr'])
def test_missing_value_raises(name):
    kw = fit_kw()
    kw.pop(name)
    with pytest.raises(ValueError, match=name):
        get_zmod(LAM, 'fitting', DAY, LON, LAT, **kw)


def test_missing_everything_lists_all():
    with pytest.raises(ValueError) as e:
        get_zmod(LAM, 'fitting', DAY, LON, LAT)
    for name in FIT_NAMES + ['solar_irr']:
        assert name in str(e.value)


@pytest.mark.parametrize('ptype', ['kelsall', 'skysurf'])
def test_fitting_args_rejected_outside_fitting(ptype):
    with pytest.raises(ValueError, match="require phase_type='fitting'"):
        get_zmod(LAM, ptype, DAY, LON, LAT, fitting_albedo=0.2)


@pytest.mark.parametrize('over', [
    dict(fitting_hong=HONG[:5]),                     # wrong Hong length
    dict(fitting_albedo=np.full(N + 1, 0.2)),        # wrong per-pixel length
    dict(fitting_albedo=np.nan),                     # non-finite
    dict(fitting_em_band=np.inf),
    dict(solar_irr=np.nan),
])
def test_bad_inputs_raise(over):
    with pytest.raises(ValueError):
        zf(**over)


# ---------------------------------------------------------- 2. slot mapping
def test_apply_fitting_writes_correct_slots():
    zpar = gz._default_zpar()
    fit = _broadcast_fitting(N, hong=HONG, albedo=0.11, em_smooth=0.22,
                             em_band=0.33, em_ring=0.44)
    rows, grp = _apply_fitting(zpar[None, :], np.zeros(N, dtype=int), fit)
    assert rows.shape[0] == 1 and np.all(grp == 0)
    r = rows[0]
    np.testing.assert_array_equal(r[183:189], HONG)
    assert np.all(r[ALB_SLOTS] == 0.11)
    assert np.all(r[EM_SMOOTH] == 0.22)
    assert np.all(r[EM_BANDS] == 0.33)
    assert np.all(r[EM_RING] == 0.44)
    touched = np.zeros(r.size, bool)
    touched[ALB_SLOTS + EM_SMOOTH + EM_BANDS + EM_RING] = True
    touched[183:189] = True
    np.testing.assert_array_equal(r[~touched], zpar[~touched])   # nothing else changed


def test_apply_fitting_agrees_with_put_zpar():
    zpar = gz._default_zpar()
    ref = put_zpar(zpar.copy(), 0, 0, 0, np.array([0.11]), det1=0,
                   hg3=HONG[None, :], E1=np.array([0.22]))[0]
    fit = _broadcast_fitting(N, hong=HONG, albedo=0.11, em_smooth=0.22,
                             em_band=0.22, em_ring=0.22)
    r = _apply_fitting(zpar[None, :], np.zeros(N, dtype=int), fit)[0][0]
    slots = ALB_SLOTS + EM_SMOOTH + EM_BANDS + EM_RING + list(range(183, 189))
    np.testing.assert_array_equal(r[slots], ref[slots])


# ------------------------------------------------- 3. reproduces 'skysurf'
@pytest.mark.parametrize('lam', [1.25, 3.0])
def test_reproduces_skysurf(lam):
    lam1 = np.array([lam])
    h = np.atleast_2d(np.asarray(get_hong_params(lam1), dtype=float)).copy()
    h[:, -3:] = h[:, 3:] * get_mult(lam1)            # same as the skysurf branch
    alb = float(np.ravel(get_albedo(lam1))[0])
    em = float(np.ravel(get_emiss(lam1))[0])
    sirr = float(np.ravel(solar_sp(lam1))[0])

    if np.isfinite(em):
        ems = dict(fitting_em_smooth=em, fitting_em_band=em, fitting_em_ring=em)
    else:
        # get_emiss NaN -> put_zpar keeps the zpar emissivities
        z = gz._default_zpar()
        bands = z[EM_BANDS]
        if not np.all(bands == bands[0]):
            pytest.skip("default band emissivities differ; one fitting_em_band can't match")
        ems = dict(fitting_em_smooth=z[37], fitting_em_band=bands[0],
                   fitting_em_ring=z[174])

    a = get_zmod(lam, 'fitting', DAY, LON, LAT, solar_irr=sirr,
                 fitting_hong=h[0], fitting_albedo=alb, **ems)
    b = get_zmod(lam, 'skysurf', DAY, LON, LAT)
    np.testing.assert_allclose(a, b, rtol=0, atol=6e-6)   # 'skysurf' is rounded


# ------------------------------------------------- 4. fitting-mode behavior
def test_output_not_rounded():
    out = zf()
    assert np.any(out != np.round(out, 5))


def test_long_wavelength_allowed_in_fitting_only():
    out = zf(lam=12.0)
    assert np.all(np.isfinite(out)) and np.all(out > 0)
    with pytest.raises(ValueError, match="3.5"):
        get_zmod(12.0, 'skysurf', DAY, LON, LAT)


def test_albedo_raises_brightness_at_short_wavelength():
    assert np.all(zf(fitting_albedo=0.30) > zf(fitting_albedo=0.20))


def test_hong_weight_raises_brightness():
    h2 = HONG.copy()
    h2[3] *= 1.2
    assert np.all(zf(fitting_hong=h2) > zf())


@pytest.mark.parametrize('name', ['fitting_em_smooth', 'fitting_em_band',
                                  'fitting_em_ring'])
def test_emissivity_matters_at_12um(name):
    lo = zf(lam=12.0, **{name: 0.8})
    hi = zf(lam=12.0, **{name: 1.0})
    assert np.all(hi >= lo) and np.any(hi > lo)


def test_per_pixel_values_grouped_correctly():
    alb = np.where(np.arange(N) % 2 == 0, 0.2, 0.3)
    mixed = zf(fitting_albedo=alb)
    expected = np.where(alb == 0.2, zf(fitting_albedo=0.2), zf(fitting_albedo=0.3))
    np.testing.assert_allclose(mixed, expected, rtol=1e-12)


# ------------------------------------------------------- 5. numba vs numpy
@pytest.mark.skipif(not gz.NUMBA_AVAILABLE, reason="numba not installed")
@pytest.mark.parametrize('lam', [1.25, 12.0])
def test_numba_matches_numpy(lam):
    gz._NUMBA_FIT_OK.clear()
    for alb in (0.25, 0.31):          # 2nd call skips numba's self-verification
        kw = fit_kw(fitting_albedo=alb)
        a = get_zmod(lam, 'fitting', DAY, LON, LAT, engine='numpy', **kw)
        b = get_zmod(lam, 'fitting', DAY, LON, LAT, engine='numba', **kw)
        assert len(gz._NUMBA_FIT_OK) == 1, "numba fell back to numpy"
        np.testing.assert_allclose(b, a, rtol=1e-10)


# ---------------------------------------------- 6. old modes are unchanged
BASE = 'zmod_baseline.npz'


@pytest.mark.skipif(not os.path.exists(BASE), reason="run make_baseline.py first")
def test_kelsall_and_skysurf_unchanged():
    base = np.load(BASE)
    for key in base.files:
        ptype, lam = key.split('_')
        np.testing.assert_array_equal(
            get_zmod(float(lam), ptype, DAY, LON, LAT), base[key], err_msg=key)

AttributeError: 'function' object has no attribute 'NUMBA_AVAILABLE'